# 02 数据清洗与整理 — 低利率时代的储蓄突围

| 项目 | 内容 |
|------|------|
| 课程 | 数据分析与经济决策（ds2026） |
| 题目 | 存款利率跌破1%，钱往哪放？ |
| 小组 | 第09组 |
| 日期 | 2026-05-23 |

本Notebook对原始数据进行清洗、格式统一和合并，为后续分析做准备。

In [ ]:
import pandas as pd
import numpy as np
import os

RAW_DIR = "data_raw"
CLEAN_DIR = "data_clean"
os.makedirs(CLEAN_DIR, exist_ok=True)

## 2.1 存款利率数据清洗

In [ ]:
df = pd.read_csv(os.path.join(RAW_DIR, "deposit_rate.csv"))
df["日期"] = pd.to_datetime(df["日期"])
df = df.sort_values("日期").reset_index(drop=True)
print(f"清洗前: {df.shape}")
print(f"缺失值: {df.isnull().sum().sum()}")
df.to_csv(os.path.join(CLEAN_DIR, "deposit_rate_clean.csv"), index=False)
print(f"清洗后: {df.shape}")
df.tail()

## 2.2 国债收益率数据清洗

In [ ]:
df = pd.read_csv(os.path.join(RAW_DIR, "bond_yield.csv"))
df["日期"] = pd.to_datetime(df["日期"])
df = df.sort_values("日期").reset_index(drop=True)
# 删除缺失值过多的行
df = df.dropna(subset=["中国国债收益率10年"])
print(f"清洗后: {df.shape}")
df.to_csv(os.path.join(CLEAN_DIR, "bond_yield_clean.csv"), index=False)
df.tail()

## 2.3 CPI数据清洗

In [ ]:
df = pd.read_csv(os.path.join(RAW_DIR, "cpi.csv"))
df["月份"] = df["月份"].str.replace("年", "-").str.replace("月份", "")
df["日期"] = pd.to_datetime(df["月份"], format="%Y-%m")
df = df.dropna(subset=["日期"]).sort_values("日期").reset_index(drop=True)
print(f"清洗后: {df.shape}")
df.to_csv(os.path.join(CLEAN_DIR, "cpi_clean.csv"), index=False)
df[["日期", "全国-当月", "全国-同比增长"]].tail()

## 2.4 M2数据清洗

In [ ]:
df = pd.read_csv(os.path.join(RAW_DIR, "money_supply.csv"))
df["月份"] = df["月份"].str.replace("年", "-").str.replace("月份", "")
df["日期"] = pd.to_datetime(df["月份"], format="%Y-%m")
df = df.dropna(subset=["日期"]).sort_values("日期").reset_index(drop=True)
print(f"清洗后: {df.shape}")
df.to_csv(os.path.join(CLEAN_DIR, "money_supply_clean.csv"), index=False)
df[["日期", "货币和准货币(M2)-数量(亿元)", "货币和准货币(M2)-同比增长"]].tail()

## 2.5 黄金数据清洗

In [ ]:
df = pd.read_csv(os.path.join(RAW_DIR, "gold_futures.csv"))
df["日期"] = pd.to_datetime(df["日期"])
df = df.sort_values("日期").reset_index(drop=True)
print(f"清洗后: {df.shape}")
df.to_csv(os.path.join(CLEAN_DIR, "gold_futures_clean.csv"), index=False)
df.tail()

## 2.6 合并核心时间序列数据

In [ ]:
# 以月度为频率，合并CPI、M2和存款利率
df_cpi = pd.read_csv(os.path.join(CLEAN_DIR, "cpi_clean.csv"), parse_dates=["日期"])
df_m2 = pd.read_csv(os.path.join(CLEAN_DIR, "money_supply_clean.csv"), parse_dates=["日期"])

merged = pd.merge(df_cpi[["日期", "全国-同比增长"]], df_m2[["日期", "货币和准货币(M2)-同比增长"]], on="日期", how="outer")
merged = merged.sort_values("日期").reset_index(drop=True)
merged.to_csv(os.path.join(CLEAN_DIR, "macro_merged.csv"), index=False)
print(f"合并后: {merged.shape}")
merged.tail()